In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import torch

In [20]:
data = pd.read_csv('./archive/dataset.csv')

In [21]:
data

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.7150,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.2670,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.1200,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.1430,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.1670,119.949,4,acoustic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113995,113995,2C3TZjDRiAzdyViavDJ217,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Sleep My Little Boy,21,384999,False,0.172,0.2350,...,-16.393,1,0.0422,0.6400,0.928000,0.0863,0.0339,125.995,5,world-music
113996,113996,1hIz5L4IB9hN3WRYPOCGPw,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Water Into Light,22,385000,False,0.174,0.1170,...,-18.318,0,0.0401,0.9940,0.976000,0.1050,0.0350,85.239,4,world-music
113997,113997,6x8ZfSoqDjuNa5SVP5QjvX,Cesária Evora,Best Of,Miss Perfumado,22,271466,False,0.629,0.3290,...,-10.895,0,0.0420,0.8670,0.000000,0.0839,0.7430,132.378,4,world-music
113998,113998,2e6sXL2bYv4bSz6VTdnfLs,Michael W. Smith,Change Your World,Friends,41,283893,False,0.587,0.5060,...,-10.889,1,0.0297,0.3810,0.000000,0.2700,0.4130,135.960,4,world-music


In [27]:
def encode_dataset(data):

    continous_features = [
        'popularity', 'danceability', 'energy', 'loudness', 
        'speechiness', 'acousticness', 'instrumentalness', 
        'liveness', 'valence', 'tempo'
    ]
    # normalize continous feature
    scaler = MinMaxScaler()
    data[continous_features] = scaler.fit_transform(data[continous_features])
    # Trigonometric Key Encoding (The Camelot Wheel)
    # This ensures Key 11 (B) and Key 0 (C) are mathematically adjacent
    data['key_sin'] = np.sin(2 * np.pi * data['key'] / 12)
    data['key_cos'] = np.cos(2 * np.pi * data['key'] / 12)
    # Categorical & Binary Encoding
    data['explicit'] = data['explicit'].astype(int)

    # Convert string genres into integer IDs for the Embedding layer later
    data['genre_id'] = data['track_genre'].astype('category').cat.codes

    feasture_cols = continous_features + ['key_sin','key_cos','mode','explicit']
    # converting values into tensor format
    x_feature = torch.tensor(data[feasture_cols].values,dtype = torch.float)
    x_genre = torch.tensor(data['genre_id'].values,dtype = torch.long)

    return x_feature,x_genre,scaler

In [28]:
encode_dataset(data)

(tensor([[ 0.7300,  0.6863,  0.4610,  ...,  0.8660,  0.0000,  0.0000],
         [ 0.5500,  0.4264,  0.1660,  ...,  0.8660,  1.0000,  0.0000],
         [ 0.5700,  0.4447,  0.3590,  ...,  1.0000,  1.0000,  0.0000],
         ...,
         [ 0.2200,  0.6386,  0.3290,  ...,  1.0000,  0.0000,  0.0000],
         [ 0.4100,  0.5959,  0.5060,  ..., -0.8660,  1.0000,  0.0000],
         [ 0.2200,  0.5340,  0.4870,  ...,  0.8660,  0.0000,  0.0000]]),
 tensor([  0,   0,   0,  ..., 113, 113, 113]),
 MinMaxScaler())

In [29]:
data

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,key_sin,key_cos,genre_id
0,0,0,Gen Hoshino,Comedy,Comedy,0.73,230666,0,0.686294,0.4610,...,0.032329,0.000001,0.3580,0.718593,0.361245,4,acoustic,0.500000,0.866025,0
1,1,0,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,0.55,149610,0,0.426396,0.1660,...,0.927711,0.000006,0.1010,0.268342,0.318397,4,acoustic,0.500000,0.866025,0
2,2,0,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,0.57,210826,0,0.444670,0.3590,...,0.210843,0.000000,0.1170,0.120603,0.313643,4,acoustic,0.000000,1.000000,0
3,3,0,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,0.71,201933,0,0.270051,0.0596,...,0.908635,0.000071,0.1320,0.143719,0.746758,3,acoustic,0.000000,1.000000,0
4,4,0,Chord Overstreet,Hold On,Hold On,0.82,198853,0,0.627411,0.4430,...,0.470884,0.000000,0.0829,0.167839,0.492863,4,acoustic,0.866025,0.500000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113995,113995,113,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Sleep My Little Boy,0.21,384999,0,0.174619,0.2350,...,0.642570,0.928000,0.0863,0.034070,0.517705,5,world-music,0.500000,-0.866025,113
113996,113996,113,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Water Into Light,0.22,385000,0,0.176650,0.1170,...,0.997992,0.976000,0.1050,0.035176,0.350242,4,world-music,0.000000,1.000000,113
113997,113997,113,Cesária Evora,Best Of,Miss Perfumado,0.22,271466,0,0.638579,0.3290,...,0.870482,0.000000,0.0839,0.746734,0.543933,4,world-music,0.000000,1.000000,113
113998,113998,113,Michael W. Smith,Change Your World,Friends,0.41,283893,0,0.595939,0.5060,...,0.382530,0.000000,0.2700,0.415075,0.558651,4,world-music,-0.500000,-0.866025,113
